In [ ]:
!pip install langchain langchain-community faiss-cpu pymupdf sentence-transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base = '/content/drive/MyDrive/PaperLens'

folders = [
    base,
    f'{base}/agents',
    f'{base}/rag',
    f'{base}/tracking',
    f'{base}/ui',
    f'{base}/data/papers',      # put your PDF papers here
    f'{base}/data/outputs',     # outputs saved here
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'✅ Created: {folder}')

print('\n🎉 PaperLens folder structure ready!')


✅ Created: /content/drive/MyDrive/PaperLens
✅ Created: /content/drive/MyDrive/PaperLens/agents
✅ Created: /content/drive/MyDrive/PaperLens/rag
✅ Created: /content/drive/MyDrive/PaperLens/tracking
✅ Created: /content/drive/MyDrive/PaperLens/ui
✅ Created: /content/drive/MyDrive/PaperLens/data/papers
✅ Created: /content/drive/MyDrive/PaperLens/data/outputs

🎉 PaperLens folder structure ready!


In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks for better retrieval."""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    return splitter.split_text(text)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model (downloads once, free)
model = SentenceTransformer('all-MiniLM-L6-v2')

def create_vector_store(chunks):
    print("Creating embeddings... (takes 1-2 mins)")
    embeddings = model.encode(chunks, show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    # Create FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    print(f"✅ Vector store created with {index.ntotal} vectors")
    return index, chunks

def search_similar(query, index, chunks, top_k=5):
    query_embedding = model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.1 MB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(api_key="ADD_YOUR_API_KEY")

def query_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",  # free model
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
def ask_paper(question, index, chunks):
    # Step 1: Find relevant chunks
    relevant_chunks = search_similar(question, index, chunks, top_k=5)
    context = "\n\n".join(relevant_chunks)

    # Step 2: Build prompt
    prompt = f"""You are an expert AI researcher.
Answer the question based ONLY on the provided context from the research paper.

Context:
{context}

Question: {question}

Give a clear, detailed answer."""

    # Step 3: Query LLM
    return query_llm(prompt)

In [ ]:
answer = ask_paper("What is the main contribution of this paper?", index, chunks)
print(answer)

The main contribution of this paper, "Attention Is All You Need," is the introduction of a new sequence transduction model, called the Transformer, which replaces complex recurrent neural networks (RNNs) with a non-recurrent architecture based on self-attention mechanisms.

The authors present a novel approach to sequence transduction that eliminates the need for RNNs and instead uses multi-head self-attention to compute context-sensitive representations of input sequences. This leads to several key advantages, including:

1. Parallelizability: The Transformer model can be parallelized at the individual token level, allowing it to scale much better than RNN-based models.
2. Increased interpretability: The attention weights can be inspected to gain insights into how the model is attending to different parts of the input sequence.
3. Simplified training: The Transformer model can be trained using a simpler and more efficient training regimen.

The key innovations of the Transformer model

In [ ]:
import gradio as gr

def upload_and_process(pdf_file):
    global index, chunks
    text = extract_text_from_pdf(pdf_file.name)
    chunks = chunk_text(text)
    index, chunks = create_vector_store(chunks)

    # Auto generate summary
    summary = ask_paper("What is this paper about? Summarize the main contributions.", index, chunks)
    return summary

def answer_question(question):
    if index is None:
        return "Please upload a paper first."
    return ask_paper(question, index, chunks)

def generate_interview_questions():
    if index is None:
        return "Please upload a paper first."
    return ask_paper("""Generate 5 technical interview questions
    with answers based on this paper.""", index, chunks)

# Initialize globals
index, chunks = None, None

with gr.Blocks(title="PaperLens") as demo:
    gr.Markdown("# 🔍 PaperLens — AI Research Paper Intelligence")

    with gr.Tab("📄 Upload & Summarize"):
        pdf_input = gr.File(label="Upload Research Paper (PDF)")
        summary_output = gr.Textbox(label="Paper Summary", lines=10)
        upload_btn = gr.Button("Analyze Paper", variant="primary")
        upload_btn.click(upload_and_process, pdf_input, summary_output)

    with gr.Tab("💬 Ask Questions"):
        question_input = gr.Textbox(label="Ask anything about the paper")
        answer_output = gr.Textbox(label="Answer", lines=8)
        ask_btn = gr.Button("Ask")
        ask_btn.click(answer_question, question_input, answer_output)

    with gr.Tab("🎯 Interview Prep"):
        gen_btn = gr.Button("Generate Interview Questions")
        questions_output = gr.Textbox(label="Interview Q&A", lines=15)
        gen_btn.click(generate_interview_questions, None, questions_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://428cfdb9390e176a53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def upload_and_process(pdf_file):
    global index, chunks
    text = extract_text_from_pdf(pdf_file.name)
    chunks = chunk_text(text)
    index, chunks = create_vector_store(chunks)
    summary = ask_paper("What is this paper about? Summarize the main contributions.", index, chunks)
    return summary

def answer_question(question):
    if index is None:
        return "Please upload a paper first."
    return ask_paper(question, index, chunks)

def generate_interview_questions():
    if index is None:
        return "Please upload a paper first."
    return ask_paper("Generate 5 technical interview questions with answers based on this paper.", index, chunks)

def generate_roadmap():
    if index is None:
        return "Please upload a paper first."
    return ask_paper("""Create a step by step implementation roadmap for a developer
    who wants to code this paper from scratch in PyTorch. Include:
    1. Prerequisites
    2. Step by step implementation plan
    3. Key components to build
    4. Common mistakes to avoid""", index, chunks)

def generate_critique():
    if index is None:
        return "Please upload a paper first."
    return ask_paper("""Critically analyze this paper:
    1. KEY STRENGTHS — what does it do well?
    2. LIMITATIONS — what are the weaknesses?
    3. FUTURE IMPROVEMENTS — what could be done next?
    4. REAL WORLD APPLICATIONS — where can this be used?""", index, chunks)

# Initialize globals
index, chunks = None, None

with gr.Blocks(title="PaperLens") as demo:
    gr.Markdown("# 🔍 PaperLens — AI Research Paper Intelligence")

    with gr.Tab("📄 Upload & Summarize"):
        pdf_input = gr.File(label="Upload Research Paper (PDF)")
        summary_output = gr.Textbox(label="Paper Summary", lines=10)
        upload_btn = gr.Button("Analyze Paper", variant="primary")
        upload_btn.click(upload_and_process, pdf_input, summary_output)

    with gr.Tab("💬 Ask Questions"):
        question_input = gr.Textbox(label="Ask anything about the paper")
        answer_output = gr.Textbox(label="Answer", lines=8)
        ask_btn = gr.Button("Ask")
        ask_btn.click(answer_question, question_input, answer_output)

    with gr.Tab("🎯 Interview Prep"):
        gen_btn = gr.Button("Generate Interview Questions")
        questions_output = gr.Textbox(label="Interview Q&A", lines=15)
        gen_btn.click(generate_interview_questions, None, questions_output)

    with gr.Tab("🗺️ Implementation Roadmap"):
        roadmap_btn = gr.Button("Generate Implementation Roadmap")
        roadmap_output = gr.Textbox(label="How to implement this paper", lines=15)
        roadmap_btn.click(generate_roadmap, None, roadmap_output)

    with gr.Tab("🔬 Compare & Critique"):
        critique_btn = gr.Button("Analyze Strengths & Weaknesses")
        critique_output = gr.Textbox(label="Critical Analysis", lines=15)
        critique_btn.click(generate_critique, None, critique_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f796bf81ae239ff7c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run this cell
import shutil
shutil.copy('/content/drive/MyDrive/PaperLens/PaperLens.ipynb',
            '/content/drive/MyDrive/PaperLens/PaperLens_backup.ipynb')
print("✅ Saved!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PaperLens/PaperLens.ipynb'